# GO-GPT inference examples

This notebook demonstrates how to use GO-GPT to predict Gene Ontology (GO) terms from protein sequences.

## Load Model

In [2]:
import sys
from pathlib import Path

# Add src to path
REPO_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from gogpt import GOGPTPredictor

# Load from HuggingFace Hub
predictor = GOGPTPredictor.from_pretrained("armansa1/gogpt-test", cache_dir="/large_storage/goodarzilab/bioreason/cache_dir")

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Loading GO-GPT model... 

...

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/1.39G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t36_3B_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded ESM2 model from facebook/esm2_t36_3B_UR50D
ESM parameters: 0.00M trainable / 2839.00M total
ESM training mode: False
number of parameters: 3100.04M
done (3109.3M parameters)


## Example proteins

In [3]:
EXAMPLES = [
    {
        "id": "A0A068CNX1",
        "organism": "Glechoma hederacea",
        "sequence": "MARLLLLLVGVLIACAAGARAGSEFLAEDNPIRQVVDGMHELESSILKAVGNSRRAFSFARFAHRYGKSYESSEEIQKRFQVYSENLRMIRSHNKKGLSYSMGVNEFSDLTWDEFKKHRLGAAQNCSATRRGNHKLTSAILPDSKDWRESGIVSPVKSQGSCGSCWTFSSTGALEAAYAQAFGKGISLSEQQLVDCAGAFNNFGCNGGLPSQAFEYIKYNGGLMTEEAYPYTGHDGECKYSSENAAVQVLDSVNITLGAEDELKHAVALVRPVSVAFEVVDGFRSYNGGVYTSTTCGSDPMDVNHAVLAVGYGVEGGVPYWLIKNSWGADWGDQGYFKMEMGKNMCGVATCASYPVVA",
    },
    {
        "id": "A0A072TK64",
        "organism": "Medicago truncatula",
        "sequence": "MEENKKTMEKSVGFTEEQDALVVKSWNAMKKNSGDLSLKFFKKILEIAPPAKQMFSFLKDSNVPLEHNPKLKPHAMSVFLMTCESAVQLRKAGKVTVRESNLKKLGATHFKTGVQDEHFEVTKQALLETIEEAIPEMWYPAMKNAWAEAHDRLANAIKAEMKEAHDQLDSANLIINMEENTGSCFTEEQEALVVKSWNAIKNNSEDLSLKFFKRIFEIAPPAKQLFSFLRDSNVPLEQNPKLKPHAMSVFLMTCESAVQLRKAGKVTVSESNLKKLGATHFKSGVKDEHFEVTKQVLLETIKEALPEMWSPAMENAWGEAHDQLANAIKAEMKKADHDHQANVEDKPSS",
    },
    {
        "id": "A0A075D5I4",
        "organism": "Rauvolfia serpentina",
        "sequence": "MAEKQQAVAEFYDNSTGAWEVFFGDHLHDGFYDPGTTATIAGSRAAVVRMIDEALRFANISDDPAKKPKTMLDVGCGIGGTCLHVAKKYGIQCKGITISSEQVKCAQGFAEEQGLEKKVSFDVGDALDMPYKDGTFDLVFTIQCIEHIQDKEKFIREMVRVAAPGAPIVIVSYAHRNLSPSEGSLKPEEKKVLKKICDNIVLSWVCSSADYVRWLTPLPVEDIKAADWTQNITPFYPLLMKEAFTWKGFTSLLMKGGWSAIKVVLAVRMMAKAADDGVLKFVAVTCRKSK",
    },
]

## Run predictions

### Plain GO terms

In [4]:
for example in EXAMPLES:
    print(f"\nProtein: {example['id']} ({example['organism']})")
    
    predictions = predictor.predict(
        sequence=example["sequence"],
        organism=example["organism"]
    )
    
    for aspect in ["MF", "BP", "CC"]:
        terms = predictions[aspect]
        if terms:
            display = ", ".join(terms[:8])
            print(f"  {aspect}: {display}")
        else:
            print(f"  {aspect}: (none)")


Protein: A0A068CNX1 (Glechoma hederacea)
  MF: GO:0003674, GO:0005488, GO:0005515
  BP: GO:0008150, GO:0032502, GO:0032501, GO:0048856, GO:0007275, GO:0009791, GO:0002164, GO:0002119
  CC: GO:0005575, GO:0110165, GO:0005622, GO:0043226, GO:0005737, GO:0005773, GO:0043229, GO:0043227

Protein: A0A072TK64 (Medicago truncatula)
  MF: GO:0003674, GO:0005488, GO:0036094, GO:0019825
  BP: GO:0008150, GO:0008152, GO:0009987, GO:0009058, GO:0044237, GO:0044281, GO:0071704, GO:0044238
  CC: GO:0005575, GO:0110165, GO:0005737, GO:0005829, GO:0005622

Protein: A0A075D5I4 (Rauvolfia serpentina)
  MF: GO:0003674, GO:0003824, GO:0016740, GO:0016741, GO:0008168, GO:0008171
  BP: GO:0008150, GO:0008152, GO:0009987, GO:0009058, GO:0044237, GO:0071704, GO:0044281, GO:1901615
  CC: GO:0005575, GO:0110165, GO:0005622, GO:0043226, GO:0043229, GO:0043227, GO:0043231, GO:0005634


### GO terms and their names

In [ ]:
# Load GO ontology to get term names
import obonet

go_ontology = obonet.read_obo(REPO_ROOT / "data" / "go-basic.obo")

def get_go_name(go_id):
    """Get the name of a GO term."""
    if go_id in go_ontology.nodes:
        return go_ontology.nodes[go_id].get("name", "Unknown")
    return "Unknown"

# Display predictions with names
for example in EXAMPLES:
    print(f"\n{'='*70}")
    print(f"Protein: {example['id']} ({example['organism']})")
    print(f"{'='*70}")
    
    predictions = predictor.predict(
        sequence=example["sequence"],
        organism=example["organism"]
    )
    
    for aspect in ["MF", "BP", "CC"]:
        aspect_name = {"MF": "Molecular Function", "BP": "Biological Process", "CC": "Cellular Component"}[aspect]
        terms = predictions[aspect]
        print(f"\n{aspect_name}:")
        if terms:
            for term in terms[:10]:
                name = get_go_name(term)
                print(f"  {term}: {name}")
        else:
            print("  (none)")